# This Section Concerns all mentioned Datasets with ANN

# Imports

In [2]:
# Standard library
import os

# Data handling
import numpy as np
import pandas as pd
from scipy import sparse

# ML utilities
from sklearn.metrics import (
    accuracy_score, 
    classification_report, 
    confusion_matrix, 
    roc_auc_score, 
    roc_curve
)

# Visualization
import seaborn as sns
import matplotlib.pyplot as plt

# Torch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader


def to_gpu_dense(X):
    """
    Converts input matrix X into a dense CuPy array safely.
    - If X is SciPy CSR/CSC/COO sparse → convert to NumPy dense → CuPy
    - If X is pandas DataFrame → convert to NumPy → CuPy
    - If X is NumPy array → CuPy
    - Never return nested object-arrays
    """

    # Case 1: SciPy sparse (CSR, CSC, COO)
    if sparse.issparse(X):
        # Convert sparse → dense NumPy → CuPy
        X_np = X.toarray().astype(np.float32)
        return cp.asarray(X_np)

    # Case 2: Pandas DataFrame
    if hasattr(X, "values"):
        return cp.asarray(X.values.astype(np.float32))

    # Case 3: NumPy array
    if isinstance(X, np.ndarray):
        return cp.asarray(X.astype(np.float32))

    # If it's already CuPy
    if isinstance(X, cp.ndarray):
        return X

    raise TypeError(f"Unsupported type passed to to_gpu_dense(): {type(X)}")

def load_split(data, split, base_path="../data/splits/"):
    """
    Loads X_train, X_test, y_train, y_test for a given dataset + split.
    Automatically detects whether features are stored as sparse (.npz)
    or dense (.csv).

    Example:
        X_train, X_test, y_train, y_test = load_split("cup98", "7030")
    """

    path = os.path.join(base_path, split)

    # ---- Load X_train ----
    npz_path = os.path.join(path, f"X_train_{data}.npz")
    csv_path = os.path.join(path, f"X_train_{data}.csv")

    if os.path.exists(npz_path):
        X_train = sparse.load_npz(npz_path)
    else:
        X_train = pd.read_csv(csv_path)

    # ---- Load X_test ----
    npz_path = os.path.join(path, f"X_test_{data}.npz")
    csv_path = os.path.join(path, f"X_test_{data}.csv")

    if os.path.exists(npz_path):
        X_test = sparse.load_npz(npz_path)
    else:
        X_test = pd.read_csv(csv_path)

    # ---- Load labels ----
    y_train = pd.read_csv(os.path.join(path, f"y_train_{data}.csv"))
    y_test  = pd.read_csv(os.path.join(path, f"y_test_{data}.csv"))

    # Convert DataFrames → Series
    y_train = y_train.iloc[:, 0]
    y_test  = y_test.iloc[:, 0]

    return X_train, X_test, y_train, y_test



def load_all_by_split(datasets, base_path="../data/splits/"):
    split_types = ["7030", "3070", "5050"]
    result = {split: {} for split in split_types}

    for split in split_types:
        print(f"\n=== Loading {split} splits ===")
        for data in datasets:
            print(f"  -> Loading {data}")
            X_train, X_test, y_train, y_test = load_split(data, split, base_path)
            result[split][data] = {
                "X_train": X_train,
                "X_test": X_test,
                "y_train": y_train,
                "y_test": y_test
            }

    return result

In [4]:
datasets = ["wine", "cup98", "customer"]

all_splits = load_all_by_split(datasets)



=== Loading 7030 splits ===
  -> Loading wine
  -> Loading cup98
  -> Loading customer

=== Loading 3070 splits ===
  -> Loading wine
  -> Loading cup98
  -> Loading customer

=== Loading 5050 splits ===
  -> Loading wine
  -> Loading cup98
  -> Loading customer


## ANN Pipeline

In [2]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------------------------------------------------
# ANN Model (1 hidden layer, ReLU)
# ----------------------------------------------------------------------
class ANN(nn.Module):
    def __init__(self, input_dim, hidden_units):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_units)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_units, 1)  # binary output
    def forward(self, x):
        x = self.relu(self.fc1(x))
        return self.fc2(x)  # logits
            

# ----------------------------------------------------------------------
# Early Stopping Helper
# ----------------------------------------------------------------------
class EarlyStopper:
    def __init__(self, patience=20):
        self.best_loss = np.inf
        self.patience = patience
        self.counter = 0
        self.best_state = None

    def check(self, loss, model):
        if loss < self.best_loss:
            self.best_loss = loss
            self.best_state = {k:v.clone() for k,v in model.state_dict().items()}
            self.counter = 0
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience


# ----------------------------------------------------------------------
# The main ANN training function (grid search)
# ----------------------------------------------------------------------
def ANN_Training_Testing(
        data,
        hidden_units_list=[1, 2, 4, 8, 32, 128],
        momentum_list=[0.0, 0.2, 0.5, 0.9],
        max_epochs=500,
        batch_size=64
    ):
    """
    Trains ANN models for all combinations of hidden_units × momentum.
    Performs validation-based early stopping and returns best model accuracy.

    Parameters:
        data: dict with "X_train", "y_train", "X_test", "y_test"
        hidden_units_list: list of hidden layer sizes
        momentum_list: list of momentum values for SGD
        max_epochs: maximum training epochs
        batch_size: dataloader batch size

    Returns:
        results: dict mapping (hidden_units, momentum) → accuracy
        best_config: tuple of (hidden_units, momentum)
    """

    # ---- Load data ----
    X_train = torch.tensor(data["X_train"], dtype=torch.float32).to(device)
    X_test  = torch.tensor(data["X_test"], dtype=torch.float32).to(device)

    y_train = torch.tensor(data["y_train"], dtype=torch.float32).to(device).reshape(-1, 1)
    y_test  = torch.tensor(data["y_test"], dtype=torch.float32).to(device).reshape(-1, 1)

    input_dim = X_train.shape[1]
    print(f"Input features: {input_dim}")

    # ---- Create loaders with 80/20 internal split for early stopping ----
    N = X_train.shape[0]
    val_size = int(0.2 * N)
    train_size = N - val_size

    train_ds, val_ds = torch.utils.data.random_split(
        TensorDataset(X_train, y_train),
        [train_size, val_size]
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    results = {}
    best_acc = -1
    best_config = None

    # ---- Grid search ----
    for hu in hidden_units_list:
        for mom in momentum_list:

            print(f"\nTraining ANN with hidden_units={hu}, momentum={mom}")

            model = ANN(input_dim=input_dim, hidden_units=hu).to(device)
            criterion = nn.BCEWithLogitsLoss()
            optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=mom)

            stopper = EarlyStopper(patience=20)

            # -------------------------
            # Training loop with early stopping
            # -------------------------
            for epoch in range(max_epochs):

                model.train()
                for xb, yb in train_loader:
                    optimizer.zero_grad()
                    logits = model(xb)
                    loss = criterion(logits, yb)
                    loss.backward()
                    optimizer.step()

                # ---- validation ----
                model.eval()
                with torch.no_grad():
                    val_losses = []
                    for xb, yb in val_loader:
                        logits = model(xb)
                        val_loss = criterion(logits, yb)
                        val_losses.append(val_loss.item())
                    val_loss_mean = np.mean(val_losses)

                # Early stopping?
                if stopper.check(val_loss_mean, model):
                    print(f"Early stopping at epoch {epoch}")
                    break

            # Restore best model state
            model.load_state_dict(stopper.best_state)

            # ---------------------------------------
            # Evaluate on test set
            # ---------------------------------------
            model.eval()
            with torch.no_grad():
                logits = model(X_test).cpu().numpy().flatten()
                preds  = (logits > 0).astype(int)
                y_true = y_test.cpu().numpy().flatten()

            acc = accuracy_score(y_true, preds)
            results[(hu, mom)] = acc

            print(f"Test Accuracy (hidden={hu}, momentum={mom}): {acc:.4f}")

            if acc > best_acc:
                best_acc = acc
                best_config = (hu, mom)
                torch.save(model.state_dict(), "best_ann_model.pt")

    print(f"\nBest ANN configuration = hidden={best_config[0]}, momentum={best_config[1]}")
    print(f"Best accuracy: {best_acc:.4f}")

    return results, best_config



# ----------------------------------------------------------------------
# Evaluation Utility (same style as your evaluate_model)
# ----------------------------------------------------------------------
def evaluate_ann_model(model, X_test, y_test, title="ANN Evaluation"):

    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test_t = torch.tensor(y_test, dtype=torch.float32).to(device).reshape(-1, 1)

    model.eval()
    with torch.no_grad():
        logits = model(X_test_t).cpu().numpy().flatten()
        preds = (logits > 0).astype(int)
        y_true = y_test_t.cpu().numpy().flatten()

    # ---------- Metrics ----------
    acc = accuracy_score(y_true, preds)
    print(f"\n=== {title} ===")
    print(f"Accuracy: {acc:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_true, preds))

    # ---------- Confusion Matrix ----------
    cm = confusion_matrix(y_true, preds)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, cmap="Blues", fmt="d")
    plt.title(f"Confusion Matrix: {title}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

    # ---------- ROC ----------
    try:
        auc = roc_auc_score(y_true, preds)
        print(f"AUC: {auc:.4f}")

        fpr, tpr, _ = roc_curve(y_true, preds)
        plt.figure(figsize=(6,5))
        plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
        plt.plot([0,1], [0,1], 'k--')
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"ROC Curve: {title}")
        plt.legend()
        plt.show()

    except Exception as e:
        print("ROC could not be computed:", e)


# 30 70 Split

In [9]:
wine_3070     = all_splits["3070"]["wine"]

X_train = wine_3070["X_train"]
X_test  = wine_3070["X_test"]
y_train = wine_3070["y_train"]
y_test  = wine_3070["y_test"]

data = {
    "X_train": X_train,
    "X_test": X_test,
    "y_train": y_train,
    "y_test": y_test
}

results, best_cfg = ANN_Training_Testing(data)


In [ ]:

cup98_3070    = all_splits["3070"]["cup98"]
customer_3070 = all_splits["3070"]["customer"]

# 70 30 Split

In [10]:
wine_7030     = all_splits["7030"]["wine"]
cup98_7030    = all_splits["7030"]["cup98"]
customer_7030 = all_splits["7030"]["customer"]


# 50 50 Split

In [11]:
wine_5050     = all_splits["5050"]["wine"]
cup98_5050    = all_splits["5050"]["cup98"]
customer_5050 = all_splits["5050"]["customer"]
